In [ ]:
import os, random, numpy as np, torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# ===== Kaggle (uploade ton kaggle.json dans /content d'abord) =====
!mkdir -p /root/.kaggle
!cp /content/kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

# ===== Téléchargement et unzip =====
!kaggle competitions download -c llm-detect-ai-generated-text -p /content/data
!unzip -o /content/data/llm-detect-ai-generated-text.zip -d /content/data

cp: cannot stat '/content/kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Traceback (most recent call last):
  File "/usr/local/bin/kaggle", line 4, in <module>
    from kaggle.cli import main
  File "/usr/local/lib/python3.11/dist-packages/kaggle/__init__.py", line 6, in <module>
    api.authenticate()
  File "/usr/local/lib/python3.11/dist-packages/kaggle/api/kaggle_api_extended.py", line 434, in authenticate
    raise IOError('Could not find {}. Make sure it\'s located in'
OSError: Could not find kaggle.json. Make sure it's located in /root/.kaggle. Or use the environment method. See setup instructions at https://github.com/Kaggle/kaggle-api/
unzip:  cannot find or open /content/data/llm-detect-ai-generated-text.zip, /content/data/llm-detect-ai-generated-text.zip.zip or /content/data/llm-detect-ai-generated-text.zip.ZIP.


In [4]:
import os
import numpy as np
import pandas as pd
from typing import Tuple

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler

from transformers import BertTokenizer, BertForSequenceClassification
from transformers import BertConfig
from transformers.models.bert.modeling_bert import BertEncoder
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import StratifiedShuffleSplit

In [5]:
# ----------------------------------
# Device
# ----------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ----------------------------------
# Paths
# ----------------------------------
DATA_DIR = ""

TRAIN_PATH  = os.path.join(DATA_DIR, "train_essays.csv")
TEST_PATH   = os.path.join(DATA_DIR, "test_essays.csv")
PROMPT_PATH = os.path.join(DATA_DIR, "train_prompts.csv")
SUB_PATH    = os.path.join(DATA_DIR, "sample_submission.csv")

src_train  = pd.read_csv(TRAIN_PATH)
src_prompt = pd.read_csv(PROMPT_PATH)
src_sub    = pd.read_csv(SUB_PATH)

# Nettoyage des noms de colonnes (par précaution)
src_train.columns = src_train.columns.str.strip()

assert "text" in src_train.columns
assert "generated" in src_train.columns

print("Distribution des labels (train):")
print(src_train["generated"].value_counts(normalize=True))

# ----------------------------------
# Model preparation
# ----------------------------------
tokenizer_save_path = "./tokenizer"
model_save_path = "./best_discriminator.pt"

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
# Discriminateur final binaire → num_labels=1 (mais on n'utilise pas la tête de classification)
pretrained_model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=1)
embedding_model = pretrained_model.bert.to(device)
embedding_model.eval()


print("TOTAL rows:", len(src_train))
print("TOTAL positives:", int(src_train["generated"].sum()))
print("TOTAL negatives:", int((src_train["generated"]==0).sum()))
print("Prompts:", src_train["prompt_id"].nunique())
print(src_train.groupby("prompt_id")["generated"].sum().sort_values(ascending=False).head(10))

Device: cuda
Distribution des labels (train):
generated
0    0.997823
1    0.002177
Name: proportion, dtype: float64


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


TOTAL rows: 1378
TOTAL positives: 3
TOTAL negatives: 1375
Prompts: 2
prompt_id
1    2
0    1
Name: generated, dtype: int64


3 positifs sur 1 378 exemples (≈ 0,22 %). Avec ça :

    Impossible d’entraîner un discriminateur supervisé “classique” (le pos_weight explose et le modèle diverge).

    Le GAN “binaire” (réels vs fakes) n’a aucun signal utile venant des positifs.

    L’AUC = 0.5 et l’AUPRC ≈ 0.0036 (≈ taux de positifs) sont “normaux”.

On doit changer la stratégie. Je te propose 3 options (dis-moi laquelle tu veux, je te livre le code complet correspondant) :

Option A — One-Class / Anomaly Detection (recommandé ici)

On oublie la classe 1 (trop rare) et on entraîne un détecteur sur la classe majoritaire (humain = 0) pour repérer les anomalies (potentiellement IA). Deux façons :

    One-Class GAN (AnoGAN / OCGAN-like) sur les embeddings BERT

        G apprend à reconstruire la “distribution humaine” (classe 0).

        Score = combinaison reconstruction error + score du D.

        Pas besoin des 3 positifs pour entraîner, on ne les utilise que pour choisir un seuil.

    Baselines rapides (pour vérifier avant)

        Isolation Forest / LOF / Mahalanobis sur les CLS embeddings BERT.

        Hyper simple, fonctionne souvent “assez bien” quand les positifs sont rarissimes.

Avantage : exploite vraiment la masse de 0.
Inconvénient : tu n’optimises pas directement la métrique Kaggle, tu relies sur un signal “anomalie”.
Option B — PU Learning (Positive–Unlabeled)

    Traite les 3 exemples “generated=1” comme positifs et tous les autres comme Non étiquetés (pas forcément négatifs).

    Utilise un nnPU / uPU risk estimator pour estimer le risque positif/négatif sans vrais négatifs.

    Implémentation plus lourde (mais je peux te coder une version PyTorch minimaliste).

Avantage : théoriquement correct.
Inconvénient : avec 3 positifs, l’estimation du prior P(y=1) est ultra instable.
Option C — Sur-échantillonnage massif + cap du pos_weight + K-fold “3 positives”

    On fait un K-fold à K=3 (un positif par fold de validation).

    On duplique agressivement les 3 positifs dans le train (oversampling) + undersampling des négatifs.

    On cappe pos_weight (ex. 10–20) pour ne pas exploser.

    On gèle BERT au début, LR très bas, gradient clipping.

    Puis (si tu veux absolument le GAN) on le lance après ce pré-entraînement.

Avantage : reste proche de l’énoncé initial (GAN binaire).
Inconvénient : très sujet à l’overfit, métriques quasi dénuées de sens (et val à 1 positif…).

In [3]:


# ----------------------------------
# Hyperparameters
# ----------------------------------
train_batch_size   = 32
val_batch_size     = 64
infer_batch_size   = 64

lr_D = 2e-4
lr_G = 2e-4
beta1 = 0.5

nz = 100                 # Dimensions du vecteur latent
num_epochs_gan = 3       # Époques de GAN (peut être augmenté)
num_epochs_sup = 2       # Époques de pré-entraînement supervisé du D

num_hidden_layers = 6    # Nombre de couches qu'on garde dans le D (sur les 12 de BERT)
train_ratio = 0.8        # split train/val
max_seq_len = 128

# ----------------------------------
# Data Preparation
# ----------------------------------
class GANDAIGDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels=None):
        self.texts = texts
        self.labels = labels  # peut être None pour test

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        if self.labels is None:
            return self.texts[idx]
        return self.texts[idx], self.labels[idx]

# Split STRATIFIÉ (important vu le fort déséquilibre)
y = src_train["generated"].values
X = src_train["text"].values

sss = StratifiedShuffleSplit(n_splits=1, test_size=1 - train_ratio, random_state=42)
train_idx, val_idx = next(sss.split(X, y))

train_set = src_train.iloc[train_idx].reset_index(drop=True)
val_set   = src_train.iloc[val_idx].reset_index(drop=True)
test_set  = pd.read_csv(TEST_PATH)

# pos_weight pour BCEWithLogitsLoss
n_pos = (train_set["generated"] == 1).sum()
n_neg = (train_set["generated"] == 0).sum()
pos_weight_tensor = torch.tensor([n_neg / max(n_pos, 1)], device=device)
print("pos_weight:", pos_weight_tensor.item(), " (#pos:", n_pos, ", #neg:", n_neg, ")")

train_dataset = GANDAIGDataset(train_set["text"].tolist(), train_set["generated"].tolist())
val_dataset   = GANDAIGDataset(val_set["text"].tolist(),   val_set["generated"].tolist())

# (Optionnel) Sampler pondéré — ici on reste sur pos_weight dans la loss
train_loader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True, drop_last=False)
val_loader   = DataLoader(val_dataset,   batch_size=val_batch_size,   shuffle=False, drop_last=False)

# ----------------------------------
# Models
# ----------------------------------
config = BertConfig(
    hidden_size=768,
    num_hidden_layers=num_hidden_layers,
    num_attention_heads=12,
    intermediate_size=3072,
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1
)

class Generator(nn.Module):
    """
    z -> (batch, seq_len, hidden_size)
    """
    def __init__(self, input_dim, seq_len=128, hidden_size=768):
        super().__init__()
        self.seq_len = seq_len
        self.hidden_size = hidden_size

        self.fc = nn.Linear(input_dim, 256 * seq_len)

        self.conv_net = nn.Sequential(
            nn.ConvTranspose1d(256, 512, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),

            nn.ConvTranspose1d(512, hidden_size, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(hidden_size),
            nn.ReLU(inplace=True),
        )

        self.bert_encoder = BertEncoder(config)

    def forward(self, x):
        # x: (batch, nz)
        bsz = x.size(0)
        x = self.fc(x)                               # (batch, 256*seq_len)
        x = x.view(bsz, 256, self.seq_len)           # (batch, 256, seq_len)
        x = self.conv_net(x)                         # (batch, 768, seq_len)
        x = x.permute(0, 2, 1).contiguous()          # (batch, seq_len, 768)

        attn_mask = torch.ones(bsz, 1, 1, self.seq_len, device=x.device)
        out = self.bert_encoder(
            hidden_states=x,
            attention_mask=attn_mask,
            return_dict=True
        )
        return out  # .last_hidden_state

class SumBertPooler(torch.nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        # hidden_states: (batch, seq_len, hidden_size)
        sum_hidden = hidden_states.sum(dim=1)        # (batch, hidden_size)
        sum_mask = sum_hidden.sum(1).unsqueeze(1)    # (batch, 1)
        sum_mask = torch.clamp(sum_mask, min=1e-9)
        mean_embeddings = sum_hidden / sum_mask
        return mean_embeddings

class Discriminator(nn.Module):
    """
    Renvoie des **logits** (pas de sigmoid dans forward).
    """
    def __init__(self):
        super().__init__()
        self.bert_encoder = BertEncoder(config)
        self.bert_encoder.layer = nn.ModuleList([
            layer for layer in pretrained_model.bert.encoder.layer[:num_hidden_layers]
        ])
        self.pooler = SumBertPooler()
        self.classifier = torch.nn.Sequential(
            nn.Linear(config.hidden_size, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(256, 1)
        )

    def forward(self, input):
        bsz, seq_len, _ = input.size()
        attn_mask = torch.ones(bsz, 1, 1, seq_len, device=input.device)
        out = self.bert_encoder(
            hidden_states=input,
            attention_mask=attn_mask,
            return_dict=True
        )
        out = self.pooler(out.last_hidden_state)  # (batch, hidden)
        out = self.classifier(out)                # (batch, 1) logits
        return out.view(-1)

# ----------------------------------
# Utils
# ----------------------------------
@torch.no_grad()
def get_embeddings(texts: list) -> torch.Tensor:
    encodings = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_seq_len,
        return_tensors="pt"
    ).to(device)

    outputs = embedding_model(
        input_ids=encodings["input_ids"],
        token_type_ids=encodings["token_type_ids"],
        attention_mask=encodings["attention_mask"]
    )
    return outputs.last_hidden_state  # (batch, seq_len, hidden)

@torch.no_grad()
def evaluate(model: nn.Module, data_loader: DataLoader) -> Tuple[float, float]:
    model.eval()
    preds = []
    trues = []
    for texts, labels in data_loader:
        emb = get_embeddings(texts)
        logits = model(emb)
        probs = torch.sigmoid(logits).cpu().numpy()
        preds.extend(probs)
        trues.extend(labels.numpy())
    auc = roc_auc_score(trues, preds)
    ap  = average_precision_score(trues, preds)
    return auc, ap

def get_model_info_dict(model, epoch, auc_score, ap_score):
    current_device = next(model.parameters()).device
    model.to('cpu')
    model_info = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'auc_score': auc_score,
        'ap_score': ap_score
    }
    model.to(current_device)
    return model_info

# ----------------------------------
# 2) Pré-entraînement supervisé du Discriminateur
# ----------------------------------
netD = Discriminator().to(device)
criterion_sup = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
optimizerD_sup = optim.Adam(netD.parameters(), lr=lr_D, betas=(beta1, 0.999))

print("==> Pré-entraînement supervisé du Discriminateur ...")
best_sup = {"epoch": -1, "auc": -1.0, "state": None}
for epoch in range(num_epochs_sup):
    netD.train()
    running_loss = 0.0
    for texts, labels in train_loader:
        labels = labels.float().to(device)
        emb = get_embeddings(texts)

        logits = netD(emb)
        loss = criterion_sup(logits, labels)
        optimizerD_sup.zero_grad()
        loss.backward()
        optimizerD_sup.step()
        running_loss += loss.item() * labels.size(0)

    auc, ap = evaluate(netD, val_loader)
    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"[Sup D] Epoch {epoch+1}/{num_epochs_sup} - loss: {epoch_loss:.4f} | AUC: {auc:.4f} | AUPRC: {ap:.4f}")
    if auc > best_sup["auc"]:
        best_sup = {"epoch": epoch, "auc": auc, "state": netD.state_dict()}

# Charger le meilleur D supervisé pour démarrer le GAN
netD.load_state_dict(best_sup["state"])
print(f"Loaded best supervised D (epoch={best_sup['epoch']}, AUC={best_sup['auc']:.4f})")

# ----------------------------------
# 3) Entraînement GAN
# ----------------------------------
netG = Generator(input_dim=nz, seq_len=max_seq_len, hidden_size=768).to(device)
criterion_gan = nn.BCEWithLogitsLoss()  # Ici, pas de pos_weight : GAN "pur"
optimizerD = optim.Adam(netD.parameters(), lr=lr_D, betas=(beta1, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=lr_G, betas=(beta1, 0.999))

def GAN_step(optimizerG, optimizerD, netG, netD, real_data, epoch, i):
    netD.train(); netG.train()

    bsz = real_data.size(0)
    # Labels GAN: vrais=1, faux=0
    real_targets = torch.ones(bsz, device=device)
    fake_targets = torch.zeros(bsz, device=device)

    # ---- D step ----
    optimizerD.zero_grad()
    logits_real = netD(real_data)
    loss_real = criterion_gan(logits_real, real_targets)

    noise = torch.randn(bsz, nz, device=device)
    fake_data = netG(noise).last_hidden_state
    logits_fake = netD(fake_data.detach())
    loss_fake = criterion_gan(logits_fake, fake_targets)

    loss_D = loss_real + loss_fake
    loss_D.backward()
    optimizerD.step()

    # ---- G step ----
    optimizerG.zero_grad()
    logits_fake_for_G = netD(fake_data)  # le G veut faire croire au D que c'est vrai => target=1
    loss_G = criterion_gan(logits_fake_for_G, real_targets)
    loss_G.backward()
    optimizerG.step()

    if i % 50 == 0:
        with torch.no_grad():
            D_x = torch.sigmoid(logits_real).mean().item()
            D_G_z1 = torch.sigmoid(logits_fake).mean().item()
            D_G_z2 = torch.sigmoid(logits_fake_for_G).mean().item()
        print('[GAN][%d/%d][%d/%d] Loss_D: %.4f Loss_G: %.4f D(x): %.4f D(G(z)): %.4f / %.4f'
              % (epoch, num_epochs_gan, i, len(train_loader), loss_D.item(), loss_G.item(), D_x, D_G_z1, D_G_z2))

model_infos = []
print("==> Entraînement GAN ...")
for epoch in range(num_epochs_gan):
    for i, (texts, labels) in enumerate(train_loader):
        with torch.no_grad():
            real_embeddings = get_embeddings(texts)
        GAN_step(
            optimizerG=optimizerG,
            optimizerD=optimizerD,
            netG=netG,
            netD=netD,
            real_data=real_embeddings,
            epoch=epoch,
            i=i
        )

    auc, ap = evaluate(netD, val_loader)
    model_infos.append(get_model_info_dict(netD, epoch, auc, ap))
    print(f"[GAN] Epoch {epoch+1}/{num_epochs_gan} - AUC: {auc:.4f} | AUPRC: {ap:.4f}")

print('Train complete！')

# ----------------------------------
# 4) Inference (meilleur modèle selon AUC)
# ----------------------------------
max_auc_model_info = max(model_infos, key=lambda x: x['auc_score'])
torch.save(max_auc_model_info, model_save_path)
print("Best GAN epoch:", max_auc_model_info["epoch"], "AUC:", max_auc_model_info["auc_score"], "AUPRC:", max_auc_model_info["ap_score"])

model = Discriminator().to(device)
model.load_state_dict(max_auc_model_info['model_state_dict'])
model.eval()

class InferenceDataset(torch.utils.data.Dataset):
    def __init__(self, texts):
        self.texts = texts
    def __getitem__(self, idx):
        return self.texts[idx]
    def __len__(self):
        return len(self.texts)

sub_dataset = InferenceDataset(test_set["text"].tolist())
inference_loader = DataLoader(sub_dataset, batch_size=infer_batch_size, shuffle=False, drop_last=False)

sub_predictions = []
with torch.no_grad():
    for texts in inference_loader:
        emb = get_embeddings(texts)
        logits = model(emb)
        probs = torch.sigmoid(logits).cpu().numpy()
        sub_predictions.extend(probs)

sub_ans_df = pd.DataFrame({
    "id": test_set["id"],
    "generated": sub_predictions
})
print(sub_ans_df.head())

sub_path = "/content/submission.csv"
sub_ans_df.to_csv(sub_path, index=False)
print("Saved submission to:", sub_path)

Device: cuda
Distribution des labels (train):
generated
0    0.997823
1    0.002177
Name: proportion, dtype: float64


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


pos_weight: 550.0  (#pos: 2 , #neg: 1100 )
==> Pré-entraînement supervisé du Discriminateur ...
[Sup D] Epoch 1/2 - loss: 156958291945.6987 | AUC: 0.5000 | AUPRC: 0.0036
[Sup D] Epoch 2/2 - loss: 126190938695.5499 | AUC: 0.5000 | AUPRC: 0.0036
Loaded best supervised D (epoch=0, AUC=0.5000)
==> Entraînement GAN ...
[GAN][0/3][0/35] Loss_D: 170581524480.0000 Loss_G: 0.0000 D(x): 1.0000 D(G(z)): 1.0000 / 1.0000
[GAN] Epoch 1/3 - AUC: 0.5000 | AUPRC: 0.0036
[GAN][1/3][0/35] Loss_D: 377853536.0000 Loss_G: 58136301568.0000 D(x): 1.0000 D(G(z)): 0.0938 / 0.0000
[GAN] Epoch 2/3 - AUC: 0.5000 | AUPRC: 0.0036
[GAN][2/3][0/35] Loss_D: 0.0000 Loss_G: 59435032576.0000 D(x): 1.0000 D(G(z)): 0.0000 / 0.0000
[GAN] Epoch 3/3 - AUC: 0.5000 | AUPRC: 0.0036
Train complete！
Best GAN epoch: 0 AUC: 0.5 AUPRC: 0.0036231884057971015
         id  generated
0  0000aaaa        1.0
1  1111bbbb        1.0
2  2222cccc        1.0
Saved submission to: /content/submission.csv
